# 公開重みのモデルで、文章を作る仕組みを観察する

**目的**：トークン分割・入力埋め込み・次の候補を見て、短い質問を試す。LLM説明の補助用（約10分、モデル取得時間は別）。共通論文の初回読解後に使う。

**費用と通信**：モデル取得にはネット接続が必要。推論は選択したMac/Colab上で実行し、OpenAI等のAPIを呼ばない。Colabの利用枠・計算資源は保証されない。

**モデル**：[Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B)、Apache 2.0。0.6Bはモデル名。固定revisionを使う。小型モデルの説明を原論文の正解として使わない。

**保存**：最後にJSONを保存する。計算機が変われば同じseedでも出力が一致するとは限らない。

## Colabで最初に行うこと

次のセルの `PREPARE_COLAB` を **True** にして実行します。公開リポジトリの固定コミットを取得し、SHA-256が一致したときだけ展開します。取得と依存のインストールには通信が必要ですが、有料APIは呼びません。既存の教材ファイルが変更されている場合は、黙って使い続けずエラーになります。

実行コードはセルを開いて確認できます。接続設定とAPIキーは教員の案内に従い、Notebook本文には保存しないでください。Google Driveへの自動保存・共有は行いません。


In [ ]:
# Colabで初回だけTrueにする。固定版の取得と依存のインストール。APIは呼ばない。
PREPARE_COLAB = False  #@param {type:"boolean"}
RELEASE_COMMIT = "699cb0f7e49f0e8dd4b2fb1dbd3f49ba4dcb27c8"
RELEASE_SHA256 = "50eacb3fe0b18b38615c3aaf2638975879a11f50454373e46e0e48d1d3870134"

# BEGIN VERIFIED BOOTSTRAP
"""Notebook冒頭へ埋め込む標準ライブラリだけの固定版取得処理。"""
from pathlib import Path, PurePosixPath
from io import BytesIO
from hashlib import sha256
from urllib.request import urlopen
import json
import re
import stat
import zipfile


def fetch_course(destination: Path, commit: str, expected_sha256: str) -> Path:
    """固定コミット・ハッシュ一致・安全な相対パスを確認して新しい教材フォルダを作る。"""
    if not re.fullmatch(r'[0-9a-f]{40}', commit) or not re.fullmatch(r'[0-9a-f]{64}', expected_sha256):
        raise ValueError('検証済みのコミットとSHA-256が設定されていません。')
    destination = destination.resolve()
    if destination.exists():
        marker = destination / '.course-release.json'
        saved = json.loads(marker.read_text()) if marker.is_file() else {}
        if saved.get('commit') != commit or saved.get('sha256') != expected_sha256 or not saved.get('files'):
            raise ValueError('既存フォルダの版が一致しません。別の空フォルダを指定してください。')
        for name, digest in saved['files'].items():
            target = (destination / name).resolve()
            if not target.is_relative_to(destination) or not target.is_file() or sha256(target.read_bytes()).hexdigest() != digest:
                raise ValueError('展開済み教材に欠落・変更があります。新しいフォルダへ取得してください。')
        return destination
    url = f'https://codeload.github.com/humansys-lab/multimodal-ai-paper-reading/zip/{commit}'
    with urlopen(url, timeout=40) as response:
        archive = response.read(20_000_001)
    if len(archive) > 20_000_000 or sha256(archive).hexdigest() != expected_sha256:
        raise ValueError('教材ZIPのサイズまたはハッシュが一致しません。展開を中止しました。')
    unpack_course(archive, destination, commit)
    files = {p.relative_to(destination).as_posix(): sha256(p.read_bytes()).hexdigest() for p in destination.rglob('*') if p.is_file()}
    (destination / '.course-release.json').write_text(json.dumps({'commit': commit, 'sha256': expected_sha256, 'files': files}))
    return destination


def unpack_course(archive: bytes, destination: Path, commit: str) -> None:
    """全メンバーの確認後に展開する。既存ファイル・リンク・不正パスを拒否する。"""
    destination = destination.resolve()
    if destination.exists():
        raise ValueError('展開先はまだ存在しないフォルダを指定してください。')
    prefix = f'multimodal-ai-paper-reading-{commit}'
    with zipfile.ZipFile(BytesIO(archive)) as bundle:
        members = []; names = set(); total = 0
        for entry in bundle.infolist():
            parts = PurePosixPath(entry.filename).parts
            if not parts or parts[0] != prefix or '..' in parts or '\\' in entry.filename or stat.S_ISLNK(entry.external_attr >> 16):
                raise ValueError('教材ZIPに不正なパスまたはリンクがあります。')
            if entry.is_dir():
                continue
            if len(parts) < 2:
                raise ValueError('教材ZIPのフォルダ構造が不正です。')
            relative = Path(*parts[1:])
            target = (destination / relative).resolve()
            if not target.is_relative_to(destination) or relative.as_posix() in names:
                raise ValueError('教材ZIPの範囲外・重複パスを検出しました。')
            names.add(relative.as_posix()); total += entry.file_size
            if total > 50_000_000:
                raise ValueError('教材ZIPの展開後サイズが上限を超えています。')
            members.append((entry, target))
        required = {'src/seminar_lab/__init__.py', 'config/course.yaml', 'materials/manifest.yaml', 'requirements-colab.txt'}
        if not required <= names:
            raise ValueError('教材ZIPに必要なコード・設定がありません。')
        destination.mkdir(parents=True)
        for entry, target in members:
            target.parent.mkdir(parents=True, exist_ok=True)
            with target.open('xb') as handle:
                handle.write(bundle.read(entry))
# END VERIFIED BOOTSTRAP

import sys, subprocess
if Path('/content').is_dir():
    ROOT = Path('/content') / ('seminar-' + RELEASE_COMMIT[:12])
    if not PREPARE_COLAB and not ROOT.exists():
        raise RuntimeError('新規ColabではPREPARE_COLABをTrueにして、このセルを実行してください。')
    ROOT = fetch_course(ROOT, RELEASE_COMMIT, RELEASE_SHA256)
    if PREPARE_COLAB:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--require-hashes',
                        '--cache-dir', str(ROOT / 'build/pip-cache'),
                        '-r', str(ROOT / 'requirements-colab.txt')], check=True)
else:
    if PREPARE_COLAB:
        raise RuntimeError('PREPARE_COLABはColab用です。ローカルではFalseにし、仮想環境を使ってください。')
    ROOT = Path.cwd().resolve()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
    if not (ROOT / 'src/seminar_lab/__init__.py').is_file():
        raise RuntimeError('教材リポジトリ、またはそのnotebooksフォルダで実行してください。')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from seminar_lab.open_weight import configure_cache, MODEL_ID, REVISION
CACHE = configure_cache(ROOT)
print('モデル:', MODEL_ID, '版:', REVISION)
print('キャッシュ:', CACHE)


## 1. 実行環境を明示する

Colabでは「ランタイムのタイプを変更」で **T4 GPU** を選びます（授業で使う機種として指定済み。割当ては画面で確認）。Colab実機検証は未完了です。Macは `mps` を選びます。CPUを試す場合は自分で `cpu` に変更します。失敗時に自動で切り替えません。

依存をまだ入れていない環境では、次の `INSTALL` を `True` に変更します。ローカルでは専用仮想環境を使用してください。

In [ ]:
INSTALL = False
if INSTALL:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--cache-dir", str(CACHE / "pip"),
                    "-r", str(ROOT / "requirements-open-weight.txt")], check=True)
else:
    print("依存のインストールは実行していません。")

In [ ]:
DEVICE = "cuda"  # Colab GPU: cuda / Apple Silicon Mac: mps / CPU: cpu
ALLOW_DOWNLOAD = False  # 最初のモデル取得時だけ明示的にTrueへ
LOAD_MODEL = False
if LOAD_MODEL:
    from seminar_lab.open_weight import load_model
    tokenizer, model = load_model(DEVICE, allow_download=ALLOW_DOWNLOAD)
    print("選択した計算機:", model.device, "数値形式:", model.dtype)
else:
    print("モデルは未読込みです。実行する計算機を選んでからLOAD_MODELをTrueへ。")

## 2. 同じ文章を、文字・ID・数値として見る

2023年講義の身近な例を使います。各トークンは単語と一致するでしょうか。数値の列は入力埋め込みの先頭4要素だけで、文脈を処理した後の表現とは異なります。数値一つに日本語の意味を割り当てないでください。

In [ ]:
OBSERVE = False
TEXT = "京都の名物は"
if OBSERVE:
    if "model" not in globals() or "tokenizer" not in globals():
        raise RuntimeError("モデルを明示的に読み込んでから実行してください。")
    from seminar_lab.open_weight import token_observation, next_token_predictions
    observation = token_observation(tokenizer, model, TEXT)
    print(json.dumps(observation, ensure_ascii=False, indent=2))
    for prefix in [TEXT, "京都の観光名所は"]:
        print(prefix, next_token_predictions(tokenizer, model, prefix))
else:
    print("観察は未実行です。")

## 3. 自分の質問で短い応答を作る

上の候補の確率は「その続きを選ぶ確率」で、事実の正しさを保証する値ではありません。質問を自分で記入し、送信前に会話テンプレートの全文を確認します。ここでは毎回新しい会話です。

思考モードは明示的に無効化します。入力512トークン、出力最大128トークンの短い観察に限定し、超過時は自動で切り詰めません。モデルが出力した文章を、内部の思考過程そのものとは扱いません。

In [ ]:
PROMPT = ""
PREVIEW = False
if PREVIEW:
    if "model" not in globals() or "tokenizer" not in globals():
        raise RuntimeError("モデルを明示的に読み込んでから実行してください。")
    from seminar_lab.open_weight import encode_prompt
    visible_template, encoded = encode_prompt(tokenizer, PROMPT)
    print(visible_template)
    print("入力トークン数:", encoded.input_ids.shape[1])

In [ ]:
RUN = False
if RUN:
    if "model" not in globals() or "tokenizer" not in globals():
        raise RuntimeError("モデルを明示的に読み込んでから実行してください。")
    from seminar_lab.open_weight import generate
    result = generate(tokenizer, model, PROMPT, max_new_tokens=96, seed=42)
    print(result["response_raw"])
    print("終了状態:", result["status"], "秒:", result["elapsed_seconds"])
    if result["status"] == "output_limit":
        print("出力上限で終了。完了した回答として扱わず、必要なら次に条件を見直します。")
else:
    print("生成は未実行です。")

## 4. 生回答と自分の理解を分けて保存する

共通論文の朝・講義後の記録へ上書きせず、仕組みの観察として別に保存します。Colabでは保存したJSONをFiles欄からダウンロードしてください。

In [ ]:
SAVE = False
if SAVE:
    if "result" not in globals():
        raise RuntimeError("保存する生成結果がありません。")
    result["student_explanation"] = ""  # 自分が説明できたこと
    result["unresolved_point"] = ""  # 確認できていないこと
    folder = ROOT / "outputs/open-weight"
    if not folder.resolve().is_relative_to(ROOT):
        raise ValueError("保存先は教材フォルダ内にしてください。")
    folder.mkdir(parents=True, exist_ok=True)
    destination = folder / (result["run_id"] + ".json")
    with destination.open("x", encoding="utf-8") as handle:
        json.dump(result, handle, ensure_ascii=False, indent=2)
    print("保存:", destination)